## Depth Anything V2 — Inference & Testing Pipeline 

End-to-end pipeline for running **Depth Anything V2** (ViT-B) on a Waymo driving segment, 
converting raw PNG outputs to calibrated NumPy depth arrays, and aligning predictions against ground-truth depth.

---

**Pipeline Stages**
1. Configure paths
2. Verify inputs
3. Download model checkpoint
4. Load & validate data
5. Sanity-check model files
6. Run inference (subprocess)
7. Verify PNG outputs
8. Convert PNGs → normalised `.npy` arrays
9. Stack arrays into a single prediction tensor
10. Align predictions with ground truth
11. Apply calibration formula
12. Save final outputs

## Configure Paths

Define all repo, segment, and output directories. Creates `raw_png/`, `raw_npy/`, and `plots/` subdirectories if they don't exist.

In [2]:
from pathlib import Path
import os

REPO_DIR = Path(r"C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026")
DA_V2_DIR = REPO_DIR / "Depth-Anything-V2"
SEGMENT_DIR = Path(r"C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_labels")


OUTPUT_DIR = REPO_DIR / "data" / "depth_anything_v2"
RAW_PNG_DIR = OUTPUT_DIR / "raw_png"
RAW_NPY_DIR = OUTPUT_DIR / "raw_npy"
PLOTS_DIR = OUTPUT_DIR / "plots"

for p in [OUTPUT_DIR, RAW_PNG_DIR, RAW_NPY_DIR, PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = SEGMENT_DIR / "images"
GT_STACK_PATH = SEGMENT_DIR / "depth_stack.npy"
GT_STEMS_PATH = SEGMENT_DIR / "depth_stack_stems.npy"

print("REPO_DIR      =", REPO_DIR)
print("DA_V2_DIR     =", DA_V2_DIR)
print("SEGMENT_DIR   =", SEGMENT_DIR)
print("IMAGES_DIR    =", IMAGES_DIR)
print("GT_STACK_PATH =", GT_STACK_PATH)
print("GT_STEMS_PATH =", GT_STEMS_PATH)
print("OUTPUT_DIR    =", OUTPUT_DIR)

REPO_DIR      = C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026
DA_V2_DIR     = C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\Depth-Anything-V2
SEGMENT_DIR   = C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_labels
IMAGES_DIR    = C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_labels\images
GT_STACK_PATH = C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_l

## Verify Input Directories

Sanity-check that the images directory and ground-truth stack files are present before proceeding.

In [3]:
print("IMAGES_DIR exists:", IMAGES_DIR.exists())
print("GT_STACK_PATH exists:", GT_STACK_PATH.exists())
print("GT_STEMS_PATH exists:", GT_STEMS_PATH.exists())

if IMAGES_DIR.exists():
    image_files = list(IMAGES_DIR.glob("*"))
    print("Number of files in images/:", len(image_files))
    print("First 5 image files:", [p.name for p in image_files[:5]])

IMAGES_DIR exists: True
GT_STACK_PATH exists: True
GT_STEMS_PATH exists: True
Number of files in images/: 197
First 5 image files: ['frame_00000.png', 'frame_00001.png', 'frame_00002.png', 'frame_00003.png', 'frame_00004.png']


## Download Model Checkpoint

Downloads the **ViT-B** checkpoint from HuggingFace if it isn't already cached locally.

In [4]:
from pathlib import Path
import urllib.request

CHECKPOINTS_DIR = DA_V2_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = CHECKPOINTS_DIR / "depth_anything_v2_vitb.pth"
CKPT_URL = "https://huggingface.co/depth-anything/Depth-Anything-V2-Base/resolve/main/depth_anything_v2_vitb.pth"

if not CKPT_PATH.exists():
    print("Downloading checkpoint...")
    urllib.request.urlretrieve(CKPT_URL, CKPT_PATH)
    print("Downloaded to:", CKPT_PATH)
else:
    print("Checkpoint already exists:", CKPT_PATH)

Checkpoint already exists: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\Depth-Anything-V2\checkpoints\depth_anything_v2_vitb.pth


## Load Images & Ground-Truth Depth

Globs input images and loads the ground-truth depth stack (`.npy`) along with its stem index. Asserts shapes are consistent.

In [5]:
import numpy as np
from pathlib import Path

image_files = sorted([p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in [".png", ".jpg", ".jpeg"]])

assert len(image_files) > 0, f"No images found in {IMAGES_DIR}"
assert GT_STACK_PATH.exists(), f"Missing {GT_STACK_PATH}"
assert GT_STEMS_PATH.exists(), f"Missing {GT_STEMS_PATH}"

gt_stack = np.load(GT_STACK_PATH, allow_pickle=True)
gt_stems = np.load(GT_STEMS_PATH, allow_pickle=True)

print("num images:", len(image_files))
print("gt_stack shape:", gt_stack.shape)
print("gt_stems shape:", gt_stems.shape)
print("first 10 image stems:", [p.stem for p in image_files[:10]])
print("first 10 gt stems:", gt_stems[:10])

num images: 197
gt_stack shape: (197, 1280, 1920)
gt_stems shape: (197,)
first 10 image stems: ['frame_00000', 'frame_00001', 'frame_00002', 'frame_00003', 'frame_00004', 'frame_00005', 'frame_00006', 'frame_00007', 'frame_00008', 'frame_00009']
first 10 gt stems: ['frame_00000' 'frame_00001' 'frame_00002' 'frame_00003' 'frame_00004'
 'frame_00005' 'frame_00006' 'frame_00007' 'frame_00008' 'frame_00009']


## Sanity-Check Model Files

Verifies that `run.py` and the checkpoints directory exist inside the Depth-Anything-V2 repo before launching inference.

In [6]:
from pathlib import Path

run_py = DA_V2_DIR / "run.py"
ckpt_dir = DA_V2_DIR / "checkpoints"

print("DA_V2_DIR exists:", DA_V2_DIR.exists())
print("run.py exists:", run_py.exists())
print("checkpoints dir exists:", ckpt_dir.exists())

if ckpt_dir.exists():
    print("checkpoint files:")
    for p in ckpt_dir.iterdir():
        print(" -", p.name)

DA_V2_DIR exists: True
run.py exists: True
checkpoints dir exists: True
checkpoint files:
 - depth_anything_v2_vitb.pth


## Run Depth Anything V2 Inference

Launches `run.py` via subprocess with `--encoder vitb`, `--pred-only`, and `--grayscale` flags. Predicted depth PNGs are written to `raw_png/`.

In [7]:
import subprocess
import sys
from pathlib import Path

run_py = DA_V2_DIR / "run.py"
assert run_py.exists(), f"Cannot find run.py at {run_py}"

cmd = [
    sys.executable, str(run_py),
    "--encoder", "vitb",
    "--img-path", str(IMAGES_DIR),
    "--outdir", str(RAW_PNG_DIR),
    "--pred-only",
    "--grayscale"
]

print("Running inference...")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    cwd=str(DA_V2_DIR),
    capture_output=True,
    text=True
)

print("return code:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError("Inference failed.")
else:
    print("Inference complete.")

Running inference...
c:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\.venv\Scripts\python.exe C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\Depth-Anything-V2\run.py --encoder vitb --img-path C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_labels\images --outdir C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\raw_png --pred-only --grayscale
return code: 0
STDOUT:
 Progress 1/197: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_labels\images\frame_00000

## Verify PNG Outputs

Lists the predicted PNG files to confirm inference produced the expected number of outputs.

In [8]:
pred_pngs = sorted(RAW_PNG_DIR.glob("*.png"))
print("num predicted pngs:", len(pred_pngs))
print("first 10 predicted files:", [p.name for p in pred_pngs[:10]])

num predicted pngs: 197
first 10 predicted files: ['frame_00000.png', 'frame_00001.png', 'frame_00002.png', 'frame_00003.png', 'frame_00004.png', 'frame_00005.png', 'frame_00006.png', 'frame_00007.png', 'frame_00008.png', 'frame_00009.png']


## Convert PNGs → Normalised NumPy Arrays

Reads each predicted grayscale PNG with OpenCV, normalises pixel values to **[0, 1]**, and saves individual `.npy` files to `raw_npy/`.

In [9]:
import cv2
import numpy as np
from pathlib import Path

depth_png_files = sorted([p for p in RAW_PNG_DIR.iterdir() if p.suffix.lower() == ".png"])
assert len(depth_png_files) > 0, f"No depth PNG files found in {RAW_PNG_DIR}"

print(f"Found {len(depth_png_files)} predicted depth PNGs")

for path in depth_png_files:
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)

    if img is None:
        print(f"Warning: failed to read {path}")
        continue

    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    img = img.astype(np.float32)

    if img.max() > img.min():
        img_normalized = (img - img.min()) / (img.max() - img.min())
    else:
        img_normalized = img.copy()

    stem = path.stem
    np.save(RAW_NPY_DIR / f"{stem}_depth.npy", img_normalized)

print("Saved individual depth arrays to:", RAW_NPY_DIR)

Found 197 predicted depth PNGs
Saved individual depth arrays to: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\raw_npy


## Stack Individual Arrays into Prediction Tensor

Loads all `*_depth.npy` files, stacks them along axis 0, and saves a single `depthanythingv2_depths.npy` tensor together with a `stems.npy` index.

In [10]:
import numpy as np
from pathlib import Path

pred_npy_files = sorted(RAW_NPY_DIR.glob("*_depth.npy"))
assert len(pred_npy_files) > 0, f"No prediction npy files found in {RAW_NPY_DIR}"

pred_stack = []
pred_stems = []

for f in pred_npy_files:
    arr = np.load(f)
    pred_stack.append(arr)
    pred_stems.append(f.stem.replace("_depth", ""))

pred_stack = np.stack(pred_stack, axis=0)
pred_stems = np.array(pred_stems)

PRED_STACK_PATH = OUTPUT_DIR / "depthanythingv2_depths.npy"
PRED_STEMS_PATH = OUTPUT_DIR / "depthanythingv2_stems.npy"

np.save(PRED_STACK_PATH, pred_stack)
np.save(PRED_STEMS_PATH, pred_stems)

print("pred_stack shape:", pred_stack.shape)
print("Saved:", PRED_STACK_PATH)
print("Saved:", PRED_STEMS_PATH)

pred_stack shape: (197, 1280, 1920)
Saved: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\depthanythingv2_depths.npy
Saved: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\depthanythingv2_stems.npy


## Align Predictions with Ground Truth

Matches prediction and GT arrays by stem name, filters to the common subset, and stacks both into aligned tensors ready for metric evaluation.

In [11]:
pred_stack = np.load(PRED_STACK_PATH, allow_pickle=True)
pred_stems = np.load(PRED_STEMS_PATH, allow_pickle=True)
gt_stack = np.load(GT_STACK_PATH, allow_pickle=True)
gt_stems = np.load(GT_STEMS_PATH, allow_pickle=True)

pred_dict = {stem: pred_stack[i] for i, stem in enumerate(pred_stems)}
gt_dict = {stem: gt_stack[i] for i, stem in enumerate(gt_stems)}

common_stems = sorted(set(pred_dict.keys()) & set(gt_dict.keys()))
assert len(common_stems) > 0, "No matching stems between prediction and ground truth"

pred_aligned = np.stack([pred_dict[s] for s in common_stems], axis=0)
gt_aligned = np.stack([gt_dict[s] for s in common_stems], axis=0)

print("Matched stems:", len(common_stems))
print("pred_aligned shape:", pred_aligned.shape)
print("gt_aligned shape:", gt_aligned.shape)
print("first 10 matched stems:", common_stems[:10])

Matched stems: 197
pred_aligned shape: (197, 1280, 1920)
gt_aligned shape: (197, 1280, 1920)
first 10 matched stems: ['frame_00000', 'frame_00001', 'frame_00002', 'frame_00003', 'frame_00004', 'frame_00005', 'frame_00006', 'frame_00007', 'frame_00008', 'frame_00009']


## Apply Calibration Formula

Defines `apply_custom_formula()` which supports three mapping types:

| `formula_type` | Expression |
|---|---|
| `None` | Identity (pass-through) |
| `"linear"` | `a·d + b` |
| `"power"` | `a·dᵇ + c` |
| `"exponential"` | `a·exp(b·d) + c` |

Set `FORMULA_TYPE` and `FORMULA_PARAMS` in the next cell to activate a calibration.

In [ ]:
## Calculating the calibration formula based out on the first 5 frames
import numpy as np
from scipy import stats
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import os
print(os.getcwd())

# Removing the outliers using iqr method and finding the standard deviation in the curve
def noiseHandler(pred40, gt40): 
    # Creating a generic mask on top of the ground truth and relative data for filtering
    groundTruth = gt40.flatten()
    predictions = pred40.flatten()

    # Using iqr to remove the outliers for the graph taking 20 outliears 
    totalBins = 20
    numberOfBins = np.linspace(0, 1, totalBins+1) 
    # Getting the index of each outlier
    numberOfBins_index = np.digitize(predictions, numberOfBins) - 1
    # Starting everything with 0 by default
    binsInlier = np.zeros(len(predictions), dtype=bool)    

    minimum_entries = 25
    for individualBins in range(totalBins):
        binsTotalEntries = (numberOfBins_index == individualBins)        
        if binsTotalEntries.sum() < minimum_entries:
            continue

        # Getting the individual entries in the area of the graph
        goundtruth_entry = groundTruth[binsTotalEntries]
        # Getting the upper and lower quartile of the curve: 25 and 75 percent
        lowerQuartile, upperQuartile = np.percentile(goundtruth_entry, [25, 75])
        # Isolating the middle entries
        iqr = upperQuartile - lowerQuartile

        binsInlier[binsTotalEntries] = (goundtruth_entry >= lowerQuartile - 1.5 * iqr) & (goundtruth_entry <= upperQuartile + 1.5*iqr)

    # Mask after getting the inliers
    predictions_cleaned, groundTruth_cleaned = predictions[binsInlier], groundTruth[binsInlier]
    # Isolating the values outside the standard deviation
    polyGraph = np.polyfit(predictions_cleaned, groundTruth_cleaned, deg=4)
    residual_enties = groundTruth_cleaned - np.polyval(polyGraph, predictions_cleaned)
    standardDevitions = np.abs(stats.zscore(residual_enties))
    # Having variable for sensitivity of the readings
    sensitivity = 4
    devitionsMask = standardDevitions < sensitivity

    p_clean, g_clean = noiseHandler(pred40, gt40)
    print(f"Points before cleaning: {len(pred40.flatten()):,}")
    print(f"Points after  cleaning: {len(p_clean):,}")

    return predictions_cleaned[devitionsMask], groundTruth_cleaned[devitionsMask]

# Function to find the specific type of relationship between relative and absolute accuracy
def model_hyperbolic(p, a, b, c):
    return a / (p + b) + c

def model_power(p, a, b, c):
    return a * np.power(np.clip(p, 1e-6, None), b) + c

def model_log(p, a, b, c):
    return a * np.log(np.clip(p + b, 1e-6, None)) + c

def model_exponential(p, a, b, c):
    return a * np.exp(b * p) + c

# Function to run all the individual models to compute the results and individual variables
def fit_all_models(p_clean, g_clean, n_sample=100_000):
    # Subsample — curve_fit doesn't need 3M points, 100k is plenty
    if len(p_clean) > n_sample:
        idx = np.random.choice(len(p_clean), n_sample, replace=False)
        p_fit = p_clean[idx]
        g_fit = g_clean[idx]
    else:
        p_fit, g_fit = p_clean, g_clean

    # Separate bounds per model — don't force hyperbolic bounds onto others
    models = {
        "hyperbolic": (model_hyperbolic, [-5.0, -0.85, 35.0],
                       ([-np.inf, -1.0 + 1e-6, -np.inf], [np.inf, -1e-6, np.inf])),
        "power":      (model_power,      [10.0, 0.5, 0.0],
                       ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])),
        "log":        (model_log,        [-20.0, 1.0, 40.0],
                       ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])),
        "exponential": (model_exponential, [2.2, 2.97, 5.6],
                       ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])),
    }

    results = {}
    for name, (fn, p0, bounds) in models.items():
        try:
            popt, _ = curve_fit(fn, p_fit, g_fit, p0=p0, bounds=bounds, maxfev=50000)
            # Evaluate metrics on full cleaned set
            g_pred = fn(p_clean, *popt)
            ss_res = np.sum((g_clean - g_pred) ** 2)
            ss_tot = np.sum((g_clean - g_clean.mean()) ** 2)
            r2   = 1 - ss_res / ss_tot
            rmse = np.sqrt(np.mean((g_clean - g_pred) ** 2))
            results[name] = {"params": popt, "R2": r2, "RMSE": rmse}
            print(f"{name:12s}  R²={r2:.4f}  RMSE={rmse:.3f}m  params={np.round(popt, 4)}")
        except Exception as e:
            print(f"{name}: failed — {e}")
    return results

# Function to extract the best median curve possible
def extract_median_curve(p_clean, g_clean, n_bins=50):
    from scipy.interpolate import UnivariateSpline
    bins = np.linspace(p_clean.min(), p_clean.max(), n_bins + 1)
    bin_centers, medians, stds, counts = [], [], [], []
    for i in range(n_bins):
        m = (p_clean >= bins[i]) & (p_clean < bins[i+1])
        if m.sum() < 10:
            continue
        bin_centers.append((bins[i] + bins[i+1]) / 2)
        medians.append(np.median(g_clean[m]))
        stds.append(np.std(g_clean[m]))
        counts.append(m.sum())
    centers = np.array(bin_centers)
    meds    = np.array(medians)
    weights = np.sqrt(np.array(counts))
    spline  = UnivariateSpline(centers, meds, w=weights/weights.max(), s=len(centers)*0.5)
    return centers, meds, np.array(stds), spline

# Finding the best plot for the graph
def plot_fit_comparison(p_raw, g_raw, p_clean, g_clean, fit_results, spline):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    x_line = np.linspace(p_clean.min(), p_clean.max(), 300)
    fn_map = {"hyperbolic": model_hyperbolic, "power": model_power, "log": model_log, "exponential": model_exponential}
    colors = {'hyperbolic': 'red', 'power': 'green', 'log': 'orange', "exponential": 'purple'}

    for ax, (p, g, title) in zip(axes, [
        (p_raw,   g_raw,   "Raw (with noise)"),
        (p_clean, g_clean, "After preprocessing")
    ]):
        # Cap scatter at 5000 points — prevents hang
        n = min(5000, len(p))
        idx = np.random.choice(len(p), n, replace=False)
        ax.scatter(p[idx], g[idx], s=1, alpha=0.3, color='steelblue', label='data')

        for name, res in fit_results.items():
            y_line = fn_map[name](x_line, *res['params'])
            ax.plot(x_line, y_line, color=colors[name], lw=2,
                    label=f"{name}  R²={res['R2']:.3f}  RMSE={res['RMSE']:.2f}m")

        ax.plot(x_line, spline(x_line), 'k--', lw=2, label='median spline')
        ax.set_xlabel("Predicted Depth (relative)")
        ax.set_ylabel("Ground Truth Depth (m)")
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.set_ylim(0, 45)

    plt.tight_layout()
    os.makedirs("plots", exist_ok=True)
    plt.savefig("plots/fit_comparison.png", dpi=150)
    plt.show()

# Function to apply custom formula for each pixels in the image
def apply_custom_formula(pred_depth, formula_type=None, params=None):
    
    if formula_type is None or params is None:
        return pred_depth.copy()

    if formula_type == "linear":
        a = params["a"]
        b = params["b"]
        return a * pred_depth + b

    elif formula_type == "power":
        a = params["a"]
        b = params["b"]
        c = params.get("c", 0.0)
        return a * np.power(np.clip(pred_depth, 1e-6, None), b) + c

    elif formula_type == "exponential":
        a = params["a"]
        b = params["b"]
        c = params.get("c", 0.0)
        return a * np.exp(b * pred_depth) + c

    elif formula_type == "hyperbolic":
        a = params["a"]
        b = params["b"]
        c = params.get("c", 0.0)
        # Clip to avoid division by zero where pred_depth == -b
        return a / np.clip(pred_depth + b, 1e-6, None) + c

    elif formula_type == "log":
        a = params["a"]
        b = params["b"]
        c = params.get("c", 0.0)
        # Clip to avoid log(0) or log of negative
        return a * np.log(np.clip(pred_depth + b, 1e-6, None)) + c

    else:
        raise ValueError(f"Unknown formula_type: {formula_type}")

### Run Calibration

Applies the configured formula frame-by-frame across `pred_aligned`. Defaults to identity mapping when `FORMULA_TYPE = None`.

In [ ]:
FORMULA_TYPE = None
FORMULA_PARAMS = None

pred5 = pred_aligned[:5]
gt5   = gt_aligned[:5]

mask = np.isfinite(pred5) & np.isfinite(gt5) & (gt5 > 0) & (gt5 < 40)

pred40 = pred5[mask]
gt40 = gt5[mask]

# Running the calibration utility
# ── Run ──────────────────────────────────────────────────────────────
p_raw = pred40.flatten()
g_raw = gt40.flatten()

fit_results = fit_all_models(p_clean, g_clean)
centers, medians, stds, spline = extract_median_curve(p_clean, g_clean)
plot_fit_comparison(p_raw, g_raw, p_clean, g_clean, fit_results, spline)

# RMSE evaluation on the full valid pixel set (0–40m)
eval_mask = np.isfinite(pred40) & np.isfinite(gt40) & (gt40 > 0) & (gt40 <= 40)
gt_eval   = gt40[eval_mask]
# --- Residuals ---
best_params = fit_results['exponential']['params']
pred_aligned_full = model_exponential(pred40[eval_mask], *best_params)
residuals = np.abs(pred_aligned_full - gt40[eval_mask])

# --- Masks ---
high_err_mask = residuals > 5.0
low_err_mask  = ~high_err_mask

num_bad  = high_err_mask.sum()
num_good = low_err_mask.sum()
total    = len(residuals)

# --- Find best model ---
fn_map = {
    "hyperbolic":  model_hyperbolic,
    "power":       model_power,
    "log":         model_log,
    "exponential": model_exponential,
}

best_name, best_rmse = None, float("inf")
pred40_eval = pred40[eval_mask] 

for name, res in fit_results.items():
    pred_transformed = fn_map[name](pred40_eval, *res["params"]) 
    rmse_full = np.sqrt(np.mean((pred_transformed - gt40[eval_mask]) ** 2))

    if rmse_full < best_rmse:
        best_rmse = rmse_full
        best_name = name

# --- Final output ---
print("\n=== FINAL SUMMARY ===")
print(f"Good pixels : {num_good:,}")
print(f"Bad pixels  : {num_bad:,}")
print(f"Total       : {total:,}")
print(f"Coverage    : {100 * num_good / total:.2f}%")

print(f"\nBest model  : {best_name}")
print(f"RMSE        : {best_rmse:.4f} m")

# Loading the remaning frames in a different stack
frame_pred = pred_aligned[5:]
FORMULA_TYPE = best_name
FORMULA_PARAMS = dict(zip(["a","b","c"], fit_results[best_name]["params"]))

pred_calibrated = np.stack(
    [apply_custom_formula(frame, FORMULA_TYPE, FORMULA_PARAMS) for frame in frame_pred],
    axis=0
)

print("pred_calibrated shape:", pred_calibrated.shape)
print("Currently using placeholder calibration (identity mapping).")

pred_calibrated shape: (197, 1280, 1920)
Currently using placeholder calibration (identity mapping).


## Save Final Outputs

Persists calibrated predictions and the aligned ground-truth tensor to disk so downstream metric scripts can load them without re-running inference.

In [20]:
CALIBRATED_PATH = OUTPUT_DIR / "depthanythingv2_calibrated_placeholder.npy"
PRED_STACK_PATH = OUTPUT_DIR / "depthanythingv2_depths.npy"
PRED_STEMS_PATH = OUTPUT_DIR / "depthanythingv2_stems.npy"
GT_ALIGNED_PATH = OUTPUT_DIR / "groundtruth_aligned.npy"

np.save(CALIBRATED_PATH, pred_calibrated)
np.save(GT_ALIGNED_PATH, gt_aligned)

print("Saved placeholder calibrated depth to:", CALIBRATED_PATH)
print("Saved aligned ground truth to:", GT_ALIGNED_PATH)

Saved placeholder calibrated depth to: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\depthanythingv2_calibrated_placeholder.npy
Saved aligned ground truth to: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\groundtruth_aligned.npy
